# Voice search

Speech query to a news or product search. Import the pipeline, load the models, check intent and query extraction, then open the Gradio demo.


## Imports

Load the record, pipeline stages, logging, and environment helper.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

if os.getenv("COLAB_RELEASE_TAG"):
    repo = Path("/content/VoiceSearch")
    if not Path("src/app/pipeline.py").is_file():
        if not (repo / "src/app/pipeline.py").is_file():
            subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/ranjanrakesh47g/VoiceSearch.git", str(repo)])
        os.chdir(repo)
    packages = [line.strip() for line in Path("requirements.txt").read_text().splitlines() if line.strip() and not line.startswith("torch")]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

In [ ]:
from src.utils.voice_search_model import VoiceSearch
from src.utils.transcribe import Transcriber
from src.utils.intent import IntentDetector
from src.utils.query import QueryExtractor
from src.utils.search import Searcher
from src.utils.logging_setup import setup_logging
from src.app.gradio_demo import build_demo
from src.app.pipeline import VoiceSearchPipeline
from dotenv import load_dotenv

/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100%|██████████| 287/287 [00:00<00:00, 3645.13it/s]


asr: distil-whisper/distil-small.en
asr_freq: 16000


Loading weights: 100%|██████████| 1259/1259 [00:00<00:00, 4602.65it/s]


asr: openai/whisper-large-v3
asr_freq: 16000


Loading weights: 100%|██████████| 723/723 [00:00<00:00, 4686.27it/s]


asr: ai-and-i-project/parakeet-tdt-0.6b-v2-hf
asr_freq: 16000
asr: amazon-transcribe


## Environment

Read `.env` and write logs under `logs/`.


In [3]:
load_dotenv()
if os.getenv("COLAB_RELEASE_TAG") and not os.getenv("BRAVE_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["BRAVE_API_KEY"] = userdata.get("BRAVE_API_KEY")
    except Exception:
        pass
setup_logging()

<Logger voice_search (INFO)>

## Models

Load the speech, intent, and named-entity models, then build the searcher.


In [4]:
transcriber = Transcriber()
intent_detector = IntentDetector()
query_extractor = QueryExtractor(intent_detector)
searcher = Searcher(query_extractor)

Loading weights: 100%|██████████| 723/723 [00:00<00:00, 10189.31it/s]
[W1006 14:17:44.543101116 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 20971520 bytes (free: 15794176, total: 8186822656).
[W1006 14:17:44.543605400 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 20971520 bytes (free: 15794176, total: 8186822656).


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 7.62 GiB of which 15.06 MiB is free. Including non-PyTorch memory, this process has 7.59 GiB memory in use. Of the allocated memory 7.24 GiB is allocated by PyTorch, and 242.54 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## Checks

Run fixed phrases and compare the detected intent and searchable query.


In [ ]:
cases = [
    ("Well tell me some news about Tom Cruise.", "news", "Tom Cruise"),
    ("What's up with Trump these days?", "news", "Trump"),
    ("Show me some t-shirts from Amazon.com", "ecommerce", "t-shirts"),
    ("get me some nyke shoes from amazon.com", "ecommerce", "nyke shoes"),
    ("find me some shoes similar to elon musk", "ecommerce", "shoes"),
    ("Tell me the latest headlines about Barack Obama.", "news", "Barack Obama"),
    ("What is happening in New York these days?", "news", "New York"),
    ("Any recent news about NASA?", "news", "NASA"),
    ("Find me wireless headphones from eBay.", "ecommerce", "wireless headphones"),
    ("Show me Adidas running shoes.", "ecommerce", "Adidas running shoes"),
    ("I want a jacket like the one Taylor Swift wore.", "ecommerce", "jacket"),
    ("What's the news about Apple's new iPhone?", "news", "Apple"),
    ("get me some addidas sneakers from flipkart", "ecommerce", "addidas sneakers"),
    ("Trump news", "news", "Trump"),
    ("looking for a cheap office chair", "ecommerce", "office chair"),
]

ok = 0
for text, actual_intent, actual_query in cases:
    record = VoiceSearch(user_query_text=text)
    intent_detector.detect_intent(record)
    query_extractor.extract_query(record)
    intent = record.intent
    query = record.searchable_query
    match = intent == actual_intent and query == actual_query
    ok += match
    print(text)
    print(f"{'ok' if match else 'WRONG'} ----- intent: {intent}, query: {query!r}")
    print()
print(f"ok: {ok}/{len(cases)}")

Batches: 100%|██████████| 1/1 [00:00<00:00, 215.27it/s]


Well tell me some news about Tom Cruise.
ok ----- intent: news, query: 'Tom Cruise'



Batches: 100%|██████████| 1/1 [00:00<00:00, 221.09it/s]


What's up with Trump these days?
ok ----- intent: news, query: 'Trump'



Batches: 100%|██████████| 1/1 [00:00<00:00, 231.93it/s]


...

get me some addidas sneakers from flipkart
ok ----- intent: ecommerce, query: 'addidas sneakers'



Batches: 100%|██████████| 1/1 [00:00<00:00, 268.99it/s]


Trump news
ok ----- intent: news, query: 'Trump'



Batches: 100%|██████████| 1/1 [00:00<00:00, 198.44it/s]

looking for a cheap office chair
ok ----- intent: ecommerce, query: 'office chair'

ok: 15/15


## Demo

Open the microphone and file-upload UI on the models loaded above.


In [ ]:
demo = build_demo(VoiceSearchPipeline(transcriber, intent_detector, query_extractor, searcher))
if os.getenv("COLAB_RELEASE_TAG"):
    demo.launch(share=True, debug=True)
else:
    demo.launch(share=True, server_port=7680)

* Running on local URL:  http://127.0.0.1:7680
* Running on public URL: https://ac4cd81ee06ade55a0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=310) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
Batches: 100%|██████████| 1/1 [00:00<00:00, 274.01it/s]
/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=300) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
Batches: 100%|██████████| 1/1 [00:00<00:00, 258.17it/s]
/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=340) to control the genera

## ASR comparison

Compare each ASR model's transcription on one microphone recording or uploaded file.


In [1]:
import os
from src.app.asr_comparison.gradio_demo import build_demo as build_compare_demo

/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 287/287 [00:00<00:00, 4234.55it/s]


asr: distil-whisper/distil-small.en
asr_freq: 16000


Loading weights: 100%|██████████| 1259/1259 [00:00<00:00, 4633.71it/s]


asr: openai/whisper-large-v3
asr_freq: 16000


Loading weights: 100%|██████████| 723/723 [00:00<00:00, 4528.96it/s]


asr: ai-and-i-project/parakeet-tdt-0.6b-v2-hf
asr_freq: 16000
asr: amazon-transcribe


In [2]:
demo = build_compare_demo()
if os.getenv("COLAB_RELEASE_TAG"):
    demo.launch(share=True, debug=True)
else:
    demo.launch(share=True, server_port=7681)

* Running on local URL:  http://127.0.0.1:7681
* Running on public URL: https://d7f6bb608a4b4c24e4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of tran

## Shut down

Shut down the demo server.


In [ ]:
demo.close()

Closing server running on port: 7680
